In [1]:
import gymnasium as gym
import numpy as np 
import math, random, os
from collections import defaultdict

In [8]:
class QLearningAgent: 
    def __init__(
        self,
        n_states, n_actions, learning_rate=0.1, gamma=0.99, 
        epsilon=1.0, epsilon_decay=0.995, min_epsilon=0.01
    ):
        self.n_states = n_states 
        self.n_actions = n_actions 
        self.lr = learning_rate
        self.gamma = gamma

        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.min_epsilon = min_epsilon

        self.q_table = np.zeros((n_states, n_actions))

    # choose action
    def choose_action(self, state):
        # exploration 
        if random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        # exploitation
        return int(np.argmax(self.q_table[state]))

    def learn(self, state, action, reward, next_state, done):
        current_q = self.q_table[state, action]
        if done:
            target_q = reward 
        else:
            base_next_q = np.max(self.q_table[next_state])
            target_q = reward + self.gamma * base_next_q

        self.q_table[state, action] += self.lr * (target_q - current_q)

    def decay_epsilon(self):
        self.epsilon = max(self.min_epsilon, self.epsilon * self.epsilon_decay)
    

In [9]:
class LineworldEnv:
    # set up the environment
    def __init__(self, size=5):
        self.size = size
        self.start_pos = 0
        self.goal_pos = size-1
        self.state = self.start_pos

        self.n_states = size 
        self.n_actions = 2 # 0 left and 1 right

    # reset the env
    def reset(self):
        self.state = self.start_pos
        return self.state
    
    # get output
    def step(self, action):
        if action == 0: self.state -= 1
        elif action == 1: self.state += 1
        else: raise ValueError("invalid action")

        self.state = max(0, min(self.state, self.size-1))
        done = self.state = self.goal_pos

        if done: reward = 10
        else: reward = -1

        info = {} # misc info about the env 
        return self.state, reward, done, info 

    # calculate reward 
    def reward(self):
        pass


In [10]:
def train(env, agent, episodes=500, max_steps=100):
    episode_rewards = []

    # loop through steps 
    for episode in range(episodes):
        # reset env for each episode 
        state = env.reset()
        total_reward = 0

        for step in range(max_steps):
            # get action 
            action = agent.choose_action(state)
            
            # take action 
            next_state, reward, done, info = env.step(action)

            # get next state and pass to model again 
            agent.learn(state, action, reward, next_state, done)
            
            state = next_state
            total_reward += reward 

            if done: break

        # train 
        agent.decay_epsilon()
        episode_rewards.append(total_reward)

        # log rewards 
        if (episode + 1) % 50 == 0:
            print(
                f"Episode {episode+1}\tReward {total_reward}\tEpsilon {agent.epsilon:.3f}")

    return episode_rewards

In [11]:
def evaluate(env, agent, episodes=5, max_steps=100):
    for episode in range(episodes):
        state = env.reset()
        total_reward = 0
        print(f"\nEpisoded {episode + 1}")

        for step in range(max_steps):
            action = int(np.argmax(agent.q_table[state]))

            next_state, reward, done, info = env.step(action)

            print(f"State {state}, Action {action}, Reward {reward}")

            state = next_state
            total_reward += reward 

            if done:
                print(f"goal reached in {step + 1} steps")
                break 

        print(f"total reward: {total_reward}")



In [12]:
if __name__ == "__main__":
    env = LineworldEnv(size=10)
    agent = QLearningAgent(
        n_states=env.n_states,
        n_actions=env.n_actions,
        learning_rate=0.1,
        gamma=0.99,
        epsilon=1.0,
        epsilon_decay=0.98,
        min_epsilon=0.01,
    )

    train(env, agent, episodes=600)

    print(f"learned q-table: {agent.q_table}")

    evaluate(env, agent)

Episode 50	Reward 10	Epsilon 0.364
Episode 100	Reward 10	Epsilon 0.133
Episode 150	Reward 10	Epsilon 0.048
Episode 200	Reward 10	Epsilon 0.018
Episode 250	Reward 10	Epsilon 0.010
Episode 300	Reward 10	Epsilon 0.010
Episode 350	Reward 10	Epsilon 0.010
Episode 400	Reward 10	Epsilon 0.010
Episode 450	Reward 10	Epsilon 0.010
Episode 500	Reward 10	Epsilon 0.010
Episode 550	Reward 10	Epsilon 0.010
Episode 600	Reward 10	Epsilon 0.010
learned q-table: [[ 9.35389181 10.        ]
 [ 0.          0.        ]
 [ 0.          0.        ]
 [ 0.          0.        ]
 [ 0.          0.        ]
 [ 0.          0.        ]
 [ 0.          0.        ]
 [ 0.          0.        ]
 [ 0.          0.        ]
 [ 0.          0.        ]]

Episoded 1
State 0, Action 1, Reward 10
goal reached in 1 steps
total reward: 10

Episoded 2
State 0, Action 1, Reward 10
goal reached in 1 steps
total reward: 10

Episoded 3
State 0, Action 1, Reward 10
goal reached in 1 steps
total reward: 10

Episoded 4
State 0, Action 1, Rewa

## Taxidriver ACtor CRitic

https://gymnasium.farama.org/environments/toy_text/taxi/

In [13]:
import torch 
import torch.nn as nn 
import torch.optim as optim 
from torch.distributions import Categorical

In [14]:
class ActorCritic(nn.Module):
    def __init__(self, n_states, n_actions, hidden_size=128):
        super().__init__()
        self.embedding = nn.Embedding(n_states, hidden_size)
        self.shared = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
        )
        
        # actor outputs action logits 
        self.actor = nn.Linear(hidden_size, n_actions)
        # critics outputs V(S) as a single scaler value 
        self.critic = nn.Linear(hidden_size, 1)
    
    def forward(self, state):
        x = self.embedding(state)
        x = self.shared(x)

        action_logits = self.actor(x)
        state_value = self.critic(x).squeeze(-1)

        return action_logits, state_value


In [15]:
class ActorCriticAgent:
    def __init__(
        self,
        n_states,
        n_actions,
        lr=1e-3,
        gamma=0.99,
        entropy_beta=0.01,
    ):
        self.gamma = gamma
        self.entropy_beta = entropy_beta

        self.model = ActorCritic(n_states, n_actions)
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)

    def choose_action(self, state):
        state_tensor = torch.tensor([state], dtype=torch.long)

        logits, value = self.model(state_tensor)

        dist = Categorical(logits=logits)
        action = dist.sample()

        log_prob = dist.log_prob(action)
        entropy = dist.entropy()

        return action.item(), log_prob.squeeze(0), value.squeeze(0), entropy.squeeze(0)

    def learn(self, log_probs, values, rewards, entropies):
        returns = []
        discounted_return = 0

        # Compute discounted returns:
        # G_t = r_t + gamma*r_{t+1} + gamma^2*r_{t+2} + ...
        for reward in reversed(rewards):
            discounted_return = reward + self.gamma * discounted_return
            returns.insert(0, discounted_return)

        returns = torch.tensor(returns, dtype=torch.float32)
        values = torch.stack(values)
        log_probs = torch.stack(log_probs)
        entropies = torch.stack(entropies)

        # Normalize returns for more stable learning.
        if len(returns) > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        # Advantage tells us whether an action was better or worse than expected.
        advantages = returns - values.detach()

        # Actor loss:
        # If advantage is positive, increase probability of action.
        # If advantage is negative, decrease probability of action.
        actor_loss = -(log_probs * advantages).mean()

        # Critic loss:
        # Make V(s) closer to the actual return.
        critic_loss = (returns - values).pow(2).mean()

        # Entropy bonus:
        # Encourages exploration.
        entropy_loss = -entropies.mean()

        total_loss = actor_loss + 0.5 * critic_loss + self.entropy_beta * entropy_loss

        self.optimizer.zero_grad()
        total_loss.backward()
        self.optimizer.step()

        return total_loss.item(), actor_loss.item(), critic_loss.item()




In [21]:
def train(
    env_name="Taxi-v4",
    episodes=3000,
    max_steps=200,
    render_every=None,
):
    env = gym.make(env_name)

    n_states = env.observation_space.n
    n_actions = env.action_space.n

    agent = ActorCriticAgent(
        n_states=n_states,
        n_actions=n_actions,
        lr=1e-3,
        gamma=0.99,
        entropy_beta=0.01,
    )

    reward_history = []

    for episode in range(1, episodes + 1):
        state, info = env.reset()

        log_probs = []
        values = []
        rewards = []
        entropies = []

        total_reward = 0

        for step in range(max_steps):
            action, log_prob, value, entropy = agent.choose_action(state)

            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            log_probs.append(log_prob)
            values.append(value)
            rewards.append(reward)
            entropies.append(entropy)

            total_reward += reward
            state = next_state

            if done:
                break

        loss, actor_loss, critic_loss = agent.learn(
            log_probs,
            values,
            rewards,
            entropies,
        )

        reward_history.append(total_reward)

        if episode % 100 == 0:
            avg_reward = sum(reward_history[-100:]) / 100

            print(
                f"Episode {episode:4d} | "
                f"Avg Reward: {avg_reward:7.2f} | "
                f"Loss: {loss:7.3f} | "
                f"Actor: {actor_loss:7.3f} | "
                f"Critic: {critic_loss:7.3f}"
            )

    env.close()

    return agent, reward_history


In [22]:
def evaluate(agent, env_name="Taxi-v4", episodes=5, max_steps=200):
    env = gym.make(env_name, render_mode="human")

    for episode in range(1, episodes + 1):
        state, info = env.reset()
        total_reward = 0

        for step in range(max_steps):
            state_tensor = torch.tensor([state], dtype=torch.long)

            with torch.no_grad():
                logits, value = agent.model(state_tensor)
                action = torch.argmax(logits, dim=-1).item()

            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            total_reward += reward
            state = next_state

            if done:
                break

        print(f"Evaluation Episode {episode} | Reward: {total_reward}")

    env.close()


In [23]:
if __name__ == "__main__":
    agent, rewards = train(
        episodes=3000,
        max_steps=200,
    )

    evaluate(agent, episodes=5)

Episode  100 | Avg Reward: -548.28 | Loss:   0.392 | Actor:  -0.099 | Critic:   1.014
Episode  200 | Avg Reward: -400.61 | Loss:   0.512 | Actor:   0.066 | Critic:   0.922
Episode  300 | Avg Reward: -292.16 | Loss:   0.388 | Actor:  -0.101 | Critic:   1.009
Episode  400 | Avg Reward: -260.28 | Loss:   0.537 | Actor:   0.108 | Critic:   0.885
Episode  500 | Avg Reward: -244.64 | Loss:   0.432 | Actor:  -0.074 | Critic:   1.039
Episode  600 | Avg Reward: -235.86 | Loss:   0.538 | Actor:  -0.032 | Critic:   1.153
Episode  700 | Avg Reward: -231.45 | Loss:   0.362 | Actor:  -0.133 | Critic:   1.014
Episode  800 | Avg Reward: -221.61 | Loss:   0.509 | Actor:  -0.018 | Critic:   1.079
Episode  900 | Avg Reward: -220.64 | Loss:   0.357 | Actor:  -0.113 | Critic:   0.964
Episode 1000 | Avg Reward: -217.61 | Loss:   0.212 | Actor:  -0.228 | Critic:   0.904
Episode 1100 | Avg Reward: -214.65 | Loss:   0.734 | Actor:   0.234 | Critic:   1.024
Episode 1200 | Avg Reward: -210.09 | Loss:   0.746 | A

In [27]:
def play_human(env_name="Taxi-v4", max_steps=200):
    env = gym.make(env_name, render_mode="ansi")

    state, info = env.reset()
    total_reward = 0

    action_meanings = {
        0: "south",
        1: "north",
        2: "east",
        3: "west",
        4: "pickup",
        5: "dropoff",
    }

    key_to_action = {
        "s": 0,  # south
        "w": 1,  # north
        "d": 2,  # east
        "a": 3,  # west
        "p": 4,  # pickup
        "o": 5,  # dropoff
    }

    print("\nControls:")
    print("w = north")
    print("s = south")
    print("a = west")
    print("d = east")
    print("p = pickup")
    print("o = dropoff")
    print("q = quit")

    for step in range(max_steps):
        print("\n" + env.render())
        print(f"Step: {step}")
        print(f"State: {state}")
        print(f"Total Reward: {total_reward}")

        user_input = input("Choose action: ").lower().strip()

        if user_input == "q":
            print("Quit game.")
            break

        if user_input not in key_to_action:
            print("Invalid action. Use w/s/a/d/p/o/q.")
            continue

        action = key_to_action[user_input]

        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        total_reward += reward

        print(f"Action: {action} ({action_meanings[action]})")
        print(f"Reward: {reward}")

        state = next_state

        if done:
            print("\n" + env.render())
            print("Episode finished.")
            print(f"Final Reward: {total_reward}")
            break

    env.close()

In [28]:
play_human()


Controls:
w = north
s = south
a = west
d = east
p = pickup
o = dropoff
q = quit

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+


Step: 0
State: 66
Total Reward: 0
Invalid action. Use w/s/a/d/p/o/q.

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+


Step: 1
State: 66
Total Reward: 0
Invalid action. Use w/s/a/d/p/o/q.

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+


Step: 2
State: 66
Total Reward: 0
Invalid action. Use w/s/a/d/p/o/q.

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+


Step: 3
State: 66
Total Reward: 0
Invalid action. Use w/s/a/d/p/o/q.

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+


Step: 4
State: 66
Total Reward: 0
Invalid action. Use w/s/a/d/p/o/q.

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+


Step: 5
State: 66
Total Reward: 0
Action: 1 (north)


KeyboardInterrupt: Interrupted by user